# Sterile neutrino (3+1) demonstrations

This notebook exercises the sterile extension of the QKE density-matrix solver (Stages A, B, C). Set `sterile_flag = True` and the solver upgrades from a 3×3 to a 4×4 Hermitian density matrix per momentum mode, adding `Δm²_41`, three active-sterile mixing angles (`θ_14`, `θ_24`, `θ_34`), one CP phase (`δ_14`), and three initial chemical potentials (`xi_nue_init`, `xi_numu_init`, `xi_nutau_init`).

Three scenarios are shown below:
1. **Stage A invariant** — `sterile_flag=True` with all mixing angles zero must reproduce the 3×3 QKE result to BBN tolerance.
2. **Dodelson-Widrow** (non-resonant) — nonzero `θ_14` with `ξ_νₑ = 0` thermalizes the sterile via collision-mediated coherences.
3. **Shi-Fuller** (MSW resonance) — small `θ_14` with nonzero `ξ_νₑ` drains the lepton asymmetry into the sterile pool as the in-medium mixing angle sweeps through resonance.

Each QKE run is ~130 s wall on a laptop (smallnet, numba JIT cached). The cells below are **not** pre-executed; run them in order after activating the environment.

In [ ]:
# Common setup
import importlib, time
import numpy as np
import PRyM.PRyM_init as PRyMini
import PRyM.PRyM_thermo as PRyMthermo
import PRyM.PRyM_main as PRyMmain

def reset_sterile():
    PRyMini.smallnet_flag = True
    PRyMini.julia_flag = False
    PRyMini.numba_flag = True
    PRyMini.compute_bckg_flag = False
    PRyMini.compute_nTOp_flag = False
    PRyMini.verbose_flag = False
    PRyMini.general_nu_flag = True
    PRyMini.boltzmann_nu_flag = True
    PRyMini.qke_density_matrix_flag = True
    PRyMini.massive_electron_flag = False
    PRyMini.sterile_flag = False
    PRyMini.Dm2_41 = 1.0
    PRyMini.theta_14 = 0.0
    PRyMini.theta_24 = 0.0
    PRyMini.theta_34 = 0.0
    PRyMini.xi_nue_init = 0.0
    PRyMini.xi_numu_init = 0.0
    PRyMini.xi_nutau_init = 0.0

def run(label):
    importlib.reload(PRyMthermo)
    t0 = time.time()
    c = PRyMmain.PRyMclass()
    res = c.PRyMresults()
    dt = time.time() - t0
    sum_ss = 0.0
    if hasattr(c, '_boltz_rho_final') and c._boltz_rho_final is not None:
        rho = c._boltz_rho_final
        if rho.shape[1] >= 4:
            sum_ss = float(rho[:, 3, :].sum())
    print(f"{label:40s} Neff={res[0]:.4f}  Yp={res[4]:.5f}  D/H={res[5]:.4f}  \u03a3\u03c1_ss={sum_ss:.3e}  ({dt:.1f}s)")
    return res, sum_ss, c

## Stage A invariant: sterile_flag=True with θ_14=0 reproduces 3×3

Guards the infrastructure: the 4×4 evolution must agree with the existing 3-flavor QKE when all active-sterile mixings are zero. Expect ΔNeff, ΔYp below 10⁻⁴.

In [ ]:
reset_sterile()
PRyMini.sterile_flag = False
res_3x3, _, _ = run('3×3 QKE reference')

reset_sterile()
PRyMini.sterile_flag = True  # theta_14=theta_24=theta_34=0
res_4x4, ss_4x4, _ = run('4×4 decoupled (θ_14=0)')

print(f"\nInvariant: \u0394Neff = {res_4x4[0] - res_3x3[0]:+.2e},  \u0394Yp = {res_4x4[4] - res_3x3[4]:+.2e}")
print(f"Sterile \u03a3\u03c1_ss = {ss_4x4:.2e}  (should be floor \u2264 1e-20)")

## Dodelson-Widrow: collision-mediated thermalization

With sin²(2θ_14) = 0.1 and Δm²_41 = 1 eV², the active-sterile coupling is large enough that the sterile thermalizes well above BBN. Expect ΔNeff ≈ +0.93 (near the +1 full-thermalization asymptote).

Physically: the evolve_step ODE for `ρ_αs` has an oscillation source `−i·s·H_αs(ρ_ss − ρ_αα)` and damping `D_αs = ½Γ_α`. In the quasi-static limit (damping faster than the integration step), the off-diagonal coherence tracks its steady state and the Sigl-Raffelt transfer

$$\Gamma_\mathrm{DW} = \frac{2\,|H_{\alpha s}|^2\,D}{D^2+\omega^2}$$

drives the diagonal `ρ_αα → ρ_ss` transfer exponentially.

In [ ]:
reset_sterile()
PRyMini.sterile_flag = True
PRyMini.Dm2_41 = 1.0
PRyMini.theta_14 = np.arcsin(np.sqrt(0.1)) / 2.0  # sin²(2θ) = 0.1
res_dw, ss_dw, _ = run(f'DW (sin²(2θ)=0.1, Δm²=1 eV²)')

print(f"\n\u0394Neff_DW = {res_dw[0] - res_3x3[0]:+.3f}   \u03a3\u03c1_ss = {ss_dw:.2f}")

## Shi-Fuller: MSW resonance driven by lepton asymmetry

A nonzero primordial `ξ_νₑ` seeds an asymmetric Fermi-Dirac for ν_e and ν̄_e and generates a matter potential `V_νₑ ∝ ξ T³`. As the universe cools the MSW condition `V_νₑ = ω_41` is met at some T_res, sweeping through momentum modes and efficiently converting ν_e into ν_s.

The resonance transfers the asymmetry itself into the sterile pool: for `ξ_νₑ = 5×10⁻²` the integrated asymmetry collapses by ~3 orders of magnitude by the end of BBN. This cell prints the residual asymmetry so the depletion is visible.

In [ ]:
def run_sf(label, xi_init):
    reset_sterile()
    PRyMini.sterile_flag = True
    PRyMini.Dm2_41 = 1.0
    PRyMini.theta_14 = np.arcsin(np.sqrt(1.0e-3)) / 2.0  # sin²(2θ) = 1e-3
    PRyMini.xi_nue_init = xi_init
    res, ss, c = run(label)
    # Asymmetry proxy: sum of y² (ρ_ee − ρ_ēē)
    n_asym = 0.0
    if c._boltz_rho_final is not None and c._boltz_solver is not None:
        y = np.asarray(c._boltz_solver.y_grid)
        rho = c._boltz_rho_final
        n_asym = float((y**2 * (rho[0, 0] - rho[1, 0])).sum())
    print(f"  residual asymmetry proxy n_\u03be\u2091 = {n_asym:+.3e}")
    return res, ss, n_asym

res_sf5, ss_sf5, nxi5 = run_sf('SF (sin²(2θ)=1e-3, ξ=5e-2)', 5.0e-2)

## Notes on parameter space

- `sin²(2θ_14) ≳ 10⁻²` typically saturates DW thermalization (ΔNeff → +1). For **non-saturating** runs, push `sin²(2θ_14) < 10⁻⁵` where the resonant enhancement from a nonzero `ξ` becomes the dominant production channel.
- The resonance temperature scales as `T_res ∝ (Δm² / ξ)^{1/4}`. For eV-scale `Δm²` and `ξ ~ 10⁻²`, this lands near T ≈ 1–3 MeV, comfortably within the BBN simulation range.
- The SF-complete matter potential (added in Stage C) includes the `trace(n_ξ)·I_active` contribution to H, which shifts `H_αα − H_ss`. In 3-flavor this trace is identically zero by ν-ν̄ symmetry; for SF it is the active-sterile MSW driver.
- For physical extrapolations to keV-scale sterile DM, change `Dm2_41` from eV² to keV² (i.e. 10⁶) and scale `ξ` accordingly (the resonance temperature moves up by `(Dm2 ratio)^{1/4}`, and the simulation has to be extended to those higher T).

See also `validation/sterile_stage_a_invariant.py`, `validation/sterile_DW_demo.py`, and `validation/sterile_SF_demo.py` for the standalone scripts these cells mirror.